In [ ]:
import jax
import jax.numpy as jnp
import custom_jax as cj

# Intro
This notebook explains what math needs to implemented to do a custom Vector Jacobian Product (vjp) that is needed for the backwards pass when advecting gradients from some output (e.g. a loss function) towards the input variables

Given a function $f_i(\vec{x})$ that calculates some output vector $\vec{y}$ with $y_i = f_i(\vec{x})$, the vjp is defined as
$$
\begin{align}
  v_k(\vec{x}, \vec{g}) &= \sum_i \frac{\partial f_i}{\partial x_k} g_i
\end{align}
$$
For the specific case of a potential or force computation we may write our function $f_i$ as a sum over pair interactions
$$
  \begin{align}
    f_i &= \sum_j m_j h (\vec{x}_i - \vec{x}_j)  - m_i h(0)  \text{\# We subtract the self-interaction explicitly} \\
    \frac{\partial f_i}{\partial x_k}  &=  \sum_j m_j  \delta_{ik} \nabla h_{ij} - m_j  \delta_{jk} \nabla h_{ij} \\
    v_k &= \sum_i \sum_j (m_j  \delta_{ik} \nabla h_{ij} - m_j  \delta_{jk} \nabla h_{ij}) g_i\\
        &= \sum_j m_j \nabla h_{kj} g_k - \sum_i m_k \nabla h_{ik} g_i \\
        &= \sum_i m_i \nabla h_{ki} g_k - m_k \nabla h_{ik} g_i \\
        &= \sum_i \nabla h_{ki} (m_i g_k + (-1)^d m_k g_i)
  \end{align}
$$
and the mass derivatives:
$$
\begin{align}
  \frac{\partial f_i}{\partial m_k}  &=  h_{ik} - \delta_{ik} h(0)\\
  v_{k,m} &= \sum_i \frac{\partial f_i}{\partial m_k} g_i  - h(0) g_k\\
  &= \sum_i h_{ik} g_i   - h(0) g_k
\end{align}
$$
where we have used the symmetry $\nabla h_{ij} = (-1)^{1+d} \nabla h_{ji}$ where $d=0$ for the potential and $d=1$ for the force

# Potential
For the potential we have
$$
\begin{align}
    h_{ki} &= - \frac{1}{|x_k - x_i|} \\
    \nabla h_{ki} &= \frac{x_k - x_i}{|x_k - x_i|^3}  \text{   \#note: gradient of potential is minus the force} \\
    v_k &= \sum_i \nabla h_{ki} (m_i g_k + m_k g_i) \\
        &= \sum_i \frac{x_k - x_i}{|x_k - x_i|^3} (m_i g_k + m_k g_i)
\end{align}
$$

In [ ]:
def potential(x, m, eps=1e-2):
    rij2 = jnp.sum((x[:, None, :] - x[None, :, :]) ** 2, axis=-1)
    rinv = 1. / jnp.sqrt(rij2 + eps**2)
    rinv = jnp.where(rij2 > 1e-10, rinv, 0.)  # mask self-interaction
    
    return -jnp.sum(rinv*m[None,:], axis=1)  #+ m / eps # remove self-interaction

def potential_fwd(x, m, eps=1e-2):
    y = potential(x, m, eps)
    return y, (x, m, eps)

def potential_bwd(res, g):
    x, m, eps = res 
    dx = x[:, None, :] - x[None, :, :]
    rij = jnp.sqrt(jnp.sum(dx**2, axis=-1)  + eps**2)
    hijgrad = dx / rij[:, :, None]**3
    v = jnp.sum(hijgrad * (g[:,None,None]*m[None,:,None] + g[None,:,None]*m[:,None,None]), axis=1)

    hij = -1. / rij
    vm = jnp.sum(hij * g[None,:], axis=1) + g / eps  # remove self-interaction
    return (v, vm, None)

potential_custom = jax.custom_vjp(potential)
potential_custom.defvjp(potential_fwd, potential_bwd)

def loss_potential(x, m):
    phi = potential(x, m)
    return jnp.sum(phi*jnp.arange(1,len(x)+1))

def loss_potential_custom(x, m):
    phi = potential_custom(x, m)
    return jnp.sum(phi*jnp.arange(1,len(x)+1))

def loss_pot_cuda(xm):
    fphi = cj.forces.ilist_fphi(xm, jnp.array([0,len(xm)]), jnp.array([[0,0]]), eps=1e-2)
    return jnp.sum(fphi[...,3]* jnp.arange(1,1+len(xm)))

x = jnp.array([[0.3, 2.0,0.3], [0.3, 4.0,-1.], [5.0, 6.0, 0.2], [0.5,0.5, 4.]])
m = jnp.array([1.0, 1.0, 1.0, 1.0])
xm = jnp.concatenate((x, m[:,None]), axis=-1)

print("Jax Auto")
print(jax.grad(loss_potential, argnums=(0,1))(x, m))
print("Jax Custom")
print(jax.grad(loss_potential_custom, argnums=(0,1))(x, m))
print("Jax FFI Custom (CUDA)")
print(jax.grad(loss_pot_cuda)(xm)[...,0:3], jax.grad(loss_pot_cuda)(xm)[...,3])

# Force
For the force we have
$$
\begin{align}
    h_{ki} &= (x_k - x_i) f_1 \\
    \nabla h_{ki} &= f_1 E + f_2 (x_k - x_i) \otimes (x_k - x_i)  \\
    f_1 &= -1/r^3 \\
    f_2 &=  3/r^5 \\
    v_k &= \sum_i \nabla h_{ki} (m_i g_k - m_k g_i) \\
        &= \sum_i f_1 (m_i g_k - m_k g_i) + f_2 (x_k - x_i) (m_i (x_k - x_i) \cdot g_k - m_k (x_k - x_i) \cdot g_i)
\end{align}
$$

In [ ]:
def force(x, m, eps=1e-2):
    dx = x[:, None, :] - x[None, :, :]
    rij2 = jnp.sum(dx ** 2, axis=-1)
    f1 = jnp.where(rij2 > 1e-8, -jnp.sqrt(rij2 + eps**2)**-3, 0.)
    hij = m[None,:,None] * dx * f1[:,:,None]
    
    return jnp.sum(hij, axis=1)

def force_fwd(x, m, eps=1e-2):
    y = force(x, m, eps)
    return y, (x, m, eps)

def force_bwd(res, g):
    x, m, eps = res
    dx = x[:, None, :] - x[None, :, :]
    rki2 = jnp.sum(dx**2, axis=-1)
    rki = jnp.sqrt(rki2  + eps**2)

    f1 = -1./rki**3
    f2 = 3./rki**5

    f1 = jnp.where(rki2 > 1e-8, f1, 0.0)
    f2 = jnp.where(rki2 > 1e-8, f2, 0.0)

    vk_i = f1[:,:,None] * (g[:,None,:]*m[None,:,None] - g[None,:,:]*m[:,None,None])

    dxgk = jnp.einsum('kid,kd->ki', dx, g)
    dxgi = jnp.einsum('kid,id->ki', dx, g)

    vk_i += f2[:,:,None] * dx * (dxgk*m[None,:] - dxgi*m[:,None])[...,None]

    vk = jnp.sum(vk_i, axis=1)

    hik = -dx * f1[:,:,None]
    vm = jnp.sum(hik * g[None,:], axis=(1,2))

    return (vk, vm, None)

force_custom = jax.custom_vjp(force)
force_custom.defvjp(force_fwd, force_bwd)

def loss_force(x, m):
    fi = force(x, m)
    return jnp.sum((fi**2)*jnp.arange(1,1+len(x))[:,None])

def loss_force_custom(x, m):
    fi = force_custom(x, m)
    return jnp.sum((fi**2)*jnp.arange(1,1+len(x))[:,None])

def loss_force_cuda(xm):
    fphi = cj.forces.ilist_fphi(xm, jnp.array([0,len(xm)]), jnp.array([[0,0]]), eps=1e-2)
    return jnp.sum(fphi[...,0:3]**2 * jnp.arange(1,1+len(xm))[:,None])

x = jnp.array([[0.2, 2.0, -1.], [0.3, 4.0, 1.], [5.0, 6.0, 2.], [0.5,0.5, 0.]])
m = jnp.array([1.0, 2.0, 0.4, 0.7])

print("Jax Auto")
print(jax.grad(loss_force, argnums=(0,1))(x, m))
print("Jax Custom")
print(jax.grad(loss_force_custom, argnums=(0,1))(x, m))
print("Jax FFI Custom (CUDA)")
xm = jnp.concatenate((x, m[:,None]), axis=-1)
print(jax.grad(loss_force_cuda)(xm)[...,0:3], jax.grad(loss_force_cuda)(xm)[...,3])